# Building a rating algorithm with `open-rating-engine`

This notebook builds a motor insurance rating algorithm from scratch with the `open_rating_engine` pricing pipeline, then scores two example policies through it.

An algorithm is a set of **layers**. Each layer computes one column (or reshapes a table), and declares which other layers it depends on. The pipeline sorts the layers into a dependency graph (DAG) and runs them in order, running independent layers in parallel.

| Stage | What it does |
|---|---|
| Data prep | Brings policy attributes down onto each vehicle, bands driver age and customer tenure, flags hire car cover |
| Exposure rating | An **ONNX** model predicts the claim cost of each vehicle |
| Claims rating | Experience-rates the policy from its own 5-year claims history, with an IBNR uplift |
| Statistical premium | Credibility-blends exposure and claims rating, then adds expense and profit loadings |
| Demand premium | Applies rating-table multipliers |
| Cap / cup | Limits how far a renewal premium can move from last year's |
| Final premium | The offered gross written premium (GWP) |
| Demand model | An **H2O** model predicts the probability the customer accepts the offered premium |
| Reporting | Offered and expected (conversion-weighted) KPIs |

**Prerequisites**

- Install the package with the ONNX, H2O and plotting extras. From the `open-rating-engine` folder: `pip install -e ".[onnx,h2o,viz]"`
- Java 8+ installed, with `JAVA_HOME` set. The H2O model is scored in a JVM.
- Run the notebook from the `.examples` folder, so the `models/` folder resolves.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from mlflow.pyfunc import PythonModelContext

from open_rating_engine import (
    ColumnCalculation,
    H2OScoringParameter,
    OnnxScoring,
    PyfuncPricingPipeline,
    RatingTablesLayer,
    TableJoin,
)

c:\git\bhavek-jamnadas\asterisk-core\.venv\Lib\site-packages\mlflow\pyfunc\model.py:209: UserWarning: Type hint used in the model's predict function is not supported for MLflow's schema validation. Type hints must be wrapped in list[...] because MLflow assumes the predict method to take multiple input instances. Specify your type hint as `list[typing.Dict[str, pandas.core.frame.DataFrame]]` for a valid signature. Remove the type hint to disable this warning. To enable validation for the input data, specify input example or model signature when logging the model. 
  func_info = _get_func_info_if_type_hint_supported(predict_attr)


## 1. Input data

The pipeline takes a dict of DataFrames, one per table. Every column is prefixed with its table name, so a column can be referenced unambiguously anywhere in the algorithm.

The example request holds two policies:

- `149110b481`: a **renewal** with 3 vehicles, two of which have hire car cover
- `0a01d5c4c2`: **new business** with 2 vehicles and no previous premium

In [2]:
DATA_DIR = Path("data").resolve() / "origination"

# Load or fetch data here
risk            = pd.read_parquet(f"{DATA_DIR}\\risk.parquet")
cover           = pd.read_parquet(f"{DATA_DIR}\\cover.parquet")
policy          = pd.read_parquet(f"{DATA_DIR}\\policy.parquet")
claims_history  = pd.read_parquet(f"{DATA_DIR}\\claims_history.parquet")


data = {
    "policy": policy,
    "risk": risk,
    "cover": cover,
    "claims_history": claims_history,
}

## 2. Data structure

The tables form a hierarchy: a policy has many vehicles (`risk`) and many years of `claims_history`, and each vehicle has many `cover` rows.

The `aggregate()` helper used inside `ColumnCalculation` expressions moves values up and down this hierarchy, such as summing vehicle claim costs onto the policy or copying the policy's transaction type onto each vehicle. It reads the join keys from the data structure, so expressions never repeat them.

In [3]:
data_structure = [
    {"name": "policy", "parent": None, "parent_key": None, "child_key": None, "cardinality": None},
    {
        "name": "risk",
        "parent": "policy",
        "parent_key": "policy.policy_number",
        "child_key": "risk.policy_number",
        "cardinality": "one_to_many",
    },
    {
        "name": "cover",
        "parent": "risk",
        "parent_key": "risk.risk_number",
        "child_key": "cover.risk_number",
        "cardinality": "one_to_many",
    },
    {
        "name": "claims_history",
        "parent": "policy",
        "parent_key": "policy.policy_number",
        "child_key": "claims_history.policy_number",
        "cardinality": "one_to_many",
    },
]

## 3. Catalog

The catalog holds static lookup tables (assumptions, reference data) shared by every layer. Layers reach them two ways:

- inside an expression, as `catalog['<name>']`
- through a `TableJoin` with `right_source="catalog"`

Because the catalog is separate from the layers, an assumption such as an expense loading can be updated without touching the algorithm.

In [4]:
catalog = {
    # Split of predicted claim cost by cover type.
    "rating_table_pred_claim_cost": pd.DataFrame({
        "cover_type": ["core", "windscreen", "hire_car"],
        "pred_claim_cost_split": [0.7, 0.1, 0.2],
    }),
    # IBNR uplift by claims-history year: recent years are still developing.
    "assumptions_ibnr": pd.DataFrame({
        "term": [-1, -2, -3, -4, -5, -6, -7],
        "ibnr_factor": [1.3, 1.1, 1.0, 1.0, 1.0, 1.0, 1.0],
    }),
    # Expense and profit loadings: fixed amounts in dollars, variable amounts as a share of risk cost.
    "assumptions_expenses": pd.DataFrame({
        "expense_type": [
            "expense_administration_fixed",
            "expense_administration_variable",
            "expense_claims_handling_fixed",
            "expense_claims_handling_variable",
            "expense_commission_fixed",
            "expense_commission_variable",
            "expense_reinsurance_expense",
            "expense_investment_income",
            "expense_target_profit_margin_percent",
        ],
        "value": [75.00, 0.02, 0.00, 0.05, 0.00, 0.05, 0.00, 0.05, 0.10],
    }),
    # Maximum premium increase (cap) and decrease (cup) at renewal, by customer tenure.
    "assumptions_cap_cup": pd.DataFrame({
        "customer_tenure_band": ["0-5", "6-10", "10+"],
        "cap_percentage": [0.5, 0.2, 0.1],
        "cup_percentage": [-0.05, -0.10, -0.20],
    }),
}

for name, table in catalog.items():
    print(name)
    display(table)

rating_table_pred_claim_cost


,cover_type,pred_claim_cost_split
0,core,0.7
1,windscreen,0.1
2,hire_car,0.2


assumptions_ibnr


,term,ibnr_factor
0,-1,1.3
1,-2,1.1
2,-3,1.0
3,-4,1.0
4,-5,1.0
5,-6,1.0
6,-7,1.0


assumptions_expenses


,expense_type,value
0,expense_administration_fixed,75.00
1,expense_administration_variable,0.02
2,expense_claims_handling_fixed,0.00
3,expense_claims_handling_variable,0.05
4,expense_commission_fixed,0.00
5,expense_commission_variable,0.05
6,expense_reinsurance_expense,0.00
7,expense_investment_income,0.05
8,expense_target_profit_margin_percent,0.10


assumptions_cap_cup


,customer_tenure_band,cap_percentage,cup_percentage
0,0-5,0.5,-0.05
1,6-10,0.2,-0.10
2,10+,0.1,-0.20


## 4. Model artifacts

Model-scoring layers don't store file paths. Each one stores an **artifact key**, and the key resolves to a file through the MLflow `PythonModelContext`. That keeps the algorithm portable: the same layers score against local files here and against downloaded registry files in production.

The `h2o_gen_model` key is reserved. The pipeline puts that jar on the JVM classpath before it loads any H2O model.

In [5]:
MODELS_DIR = Path("models").resolve()

artifacts = {
    "motor_risk_cost_onnx": MODELS_DIR / "motor_risk_cost_onnx" / "pipeline_onnx.onnx",
    "motor_demand_mojo": MODELS_DIR / "motor_demand_h2o" / "model.zip",
    "h2o_gen_model": MODELS_DIR / "motor_demand_h2o" / "h2o-genmodel.jar",
}
missing = [key for key, path in artifacts.items() if not path.exists()]
assert not missing, f"Model files not found for {missing}. Run this notebook from open-rating-engine/.examples."

context = PythonModelContext(
    artifacts={key: str(path) for key, path in artifacts.items()},
    model_config={
        # "local" makes predict() take and return DataFrames.
        "environment": "local",
        # Options for the JVM that scores the H2O model. Must not be empty: the JVM rejects a blank option.
        "java_options": "-Xmx1g",
    },
)

## 5. Layers

Each section below builds a list of layers for one stage of the algorithm. Every layer has:

- `name`: a unique identifier. Here it matches the column the layer writes.
- `dependencies`: names of the layers that must run first. The pipeline does not read expressions to work this out, so **every column produced by another layer must be listed**. Columns supplied in the input data need no dependency.

`ColumnCalculation` expressions are Python strings evaluated with these names in scope:

| Name | Meaning |
|---|---|
| `df` | The layer's target table |
| `data` | Every table, keyed by name |
| `catalog` | The catalog tables |
| `np`, `pd` | NumPy and pandas |
| `aggregate(source, agg_func, source_column)` | Aggregates `source` onto the target table using the data structure keys |

### 5.1 Data prep

Copies policy attributes onto each vehicle, bands driver age and customer tenure, and flags vehicles with hire car cover.

In [6]:
data_prep = [
    ColumnCalculation(
        name="risk.transaction_type",
        target_table="risk",
        output_column="risk.transaction_type",
        calculation="aggregate(source='policy', source_column='policy.transaction_type', agg_func='first')",
        description="Policy transaction type (NB / RN) copied onto each vehicle.",
    ),
    ColumnCalculation(
        name="risk.customer_tenure",
        target_table="risk",
        output_column="risk.customer_tenure",
        calculation="aggregate(source='policy', source_column='policy.customer_tenure', agg_func='first')",
        description="Customer tenure copied onto each vehicle.",
    ),
    ColumnCalculation(
        name="risk.customer_tenure_band",
        target_table="risk",
        output_column="risk.customer_tenure_band",
        calculation="pd.cut(df['risk.customer_tenure'], bins=[-1, 5, 10, np.inf], labels=['0-5', '6-10', '10+'])",
        dependencies=["risk.customer_tenure"],
        description="Customer tenure band.",
    ),
    ColumnCalculation(
        name="risk.risk_age_band",
        target_table="risk",
        output_column="risk.risk_age_band",
        calculation=(
            "pd.cut(df['risk.driver_youngest_age'], bins=list(range(15, 66, 10)) + [np.inf], "
            "labels=['15-24', '25-34', '35-44', '45-54', '55-64', '65+'])"
        ),
        description="Youngest driver age band.",
    ),
    ColumnCalculation(
        name="cover.cover_flag_hire_car",
        target_table="cover",
        output_column="cover.cover_flag_hire_car",
        calculation="np.where(df['cover.cover_type'] == 'hire_car', 1, 0)",
        description="Flags hire car cover rows.",
    ),
    ColumnCalculation(
        name="risk.cover_flag_hire_car",
        target_table="risk",
        output_column="risk.cover_flag_hire_car",
        calculation="aggregate(source='cover', source_column='cover.cover_flag_hire_car', agg_func='max')",
        dependencies=["cover.cover_flag_hire_car"],
        description="1 if the vehicle has hire car cover.",
    ),
]

### 5.2 Exposure rating

An ONNX model predicts each vehicle's claim cost from its rating factors. `column_map` maps the ONNX graph's input names (keys) to columns on the target table (values), so the model can be trained on different column names from the ones in the live data. Each input is cast to the dtype the graph declares.

The vehicle predictions are then summed onto the policy.

In [7]:
exposure_rating = [
    OnnxScoring(
        name="risk.pred_claim_cost",
        target_table="risk",
        output_column="risk.pred_claim_cost",
        model_artifact_key="motor_risk_cost_onnx",
        column_map={
            "driver_youngest_age": "risk.driver_youngest_age",
            "vehicle_market_value": "risk.vehicle_market_value",
            "risk_usage": "risk.vehicle_usage",
            "transaction_type": "risk.transaction_type",
            "claims_history_claim_count_5_years": "risk.claims_history_claim_count_5_years",
            "coverage_hire_car": "risk.cover_flag_hire_car",
            "driver_youngest_occupation": "risk.driver_youngest_occupation",
            "location_geoid": "risk.location_geoid",
            "location_population": "risk.location_population",
            "location_proportion_white": "risk.location_proportion_white",
            "location_statefp": "risk.location_statefp",
            "risk_average_milage": "risk.average_mileage",
            "risk_basic_excess": "risk.basic_excess",
            "risk_customer_tenure": "risk.customer_tenure",
            "vehicle_age": "risk.vehicle_age",
            "vehicle_fuel_type": "risk.vehicle_fuel_type",
            "vehicle_make": "risk.vehicle_make",
        },
        dependencies=["risk.transaction_type", "risk.cover_flag_hire_car", "risk.customer_tenure"],
        description="Claim cost scored from an ONNX model.",
    ),
    ColumnCalculation(
        name="policy.pred_claim_cost",
        target_table="policy",
        output_column="policy.pred_claim_cost",
        calculation="aggregate(source='risk', source_column='risk.pred_claim_cost', agg_func='sum')",
        dependencies=["risk.pred_claim_cost"],
        description="Exposure-rated claim cost for the policy.",
    ),
]

### 5.3 Claims rating

Rates the policy on its own claims experience. A `TableJoin` brings the IBNR factor from the catalog onto each claims-history year, so recent years still developing are uplifted to their expected ultimate cost. The ultimate cost per vehicle-year is then applied to the policy's current vehicle count.

In [8]:
claims_rating = [
    ColumnCalculation(
        name="policy.vehicle_count",
        target_table="policy",
        output_column="policy.vehicle_count",
        calculation="aggregate(source='risk', source_column='risk.risk_number', agg_func='count')",
        description="Number of vehicles on the policy.",
    ),
    TableJoin(
        name="claims_history_ibnr",
        left_table="claims_history",
        right_table="assumptions_ibnr",
        right_source="catalog",
        left_on="claims_history.term",
        right_on="term",
        columns=["ibnr_factor"],
        how="left",
        description="Join IBNR factor onto claims_history.",
    ),
    ColumnCalculation(
        name="claims_history.incurred_ultimate",
        target_table="claims_history",
        output_column="claims_history.incurred_ultimate",
        calculation="df['claims_history.incurred'] * df['ibnr_factor']",
        dependencies=["claims_history_ibnr"],
        description="Ultimate incurred claims.",
    ),
    ColumnCalculation(
        name="claims_history.cost_per_vehicle_ultimate",
        target_table="claims_history",
        output_column="claims_history.cost_per_vehicle_ultimate",
        calculation="df['claims_history.incurred_ultimate'] / df['claims_history.average_number_vehicles']",
        dependencies=["claims_history.incurred_ultimate"],
        description="Ultimate incurred claims per vehicle, by year.",
    ),
    ColumnCalculation(
        name="policy.incurred_ultimate",
        target_table="policy",
        output_column="policy.incurred_ultimate",
        calculation="aggregate(source='claims_history', source_column='claims_history.incurred_ultimate', agg_func='sum')",
        dependencies=["claims_history.incurred_ultimate"],
    ),
    ColumnCalculation(
        name="policy.average_number_vehicles",
        target_table="policy",
        output_column="policy.average_number_vehicles",
        calculation="aggregate(source='claims_history', source_column='claims_history.average_number_vehicles', agg_func='sum')",
        description="Vehicle-years of exposure in the claims history.",
    ),
    ColumnCalculation(
        name="policy.claim_count",
        target_table="policy",
        output_column="policy.claim_count",
        calculation="aggregate(source='claims_history', source_column='claims_history.claim_count', agg_func='sum')",
    ),
    ColumnCalculation(
        name="policy.cost_per_vehicle",
        target_table="policy",
        output_column="policy.cost_per_vehicle",
        calculation="df['policy.incurred_ultimate'] / df['policy.average_number_vehicles']",
        dependencies=["policy.incurred_ultimate", "policy.average_number_vehicles"],
        description="Claims-history cost per vehicle-year.",
    ),
    ColumnCalculation(
        name="policy.claims_rated_pred_claim_cost",
        target_table="policy",
        output_column="policy.claims_rated_pred_claim_cost",
        calculation="df['policy.cost_per_vehicle'] * df['policy.vehicle_count']",
        dependencies=["policy.cost_per_vehicle", "policy.vehicle_count"],
        description="Claims-rated claim cost for the policy.",
    ),
]

### 5.4 Statistical premium

Blends the two claim cost estimates with a credibility weight `Z = n / (n + 100)`, where `n` is the policy's claim count, so policies with more claims history lean more on their own experience. The blended policy cost is allocated back to each vehicle in proportion to its exposure-rated cost, and the expense and profit loadings from the catalog are added.

The `expense()` helper only builds expression text. The layers still hold plain strings, so the algorithm stays serialisable.

In [9]:
def expense(expense_type: str) -> str:
    """Expression text that looks up one value from the assumptions_expenses catalog table."""
    return f"catalog['assumptions_expenses'].set_index('expense_type').loc['{expense_type}', 'value']"


RISK_COST = "df['risk.premium_stat_prem_risk_cost']"

statistical_premium = [
    ColumnCalculation(
        name="policy.credibility_weighting",
        target_table="policy",
        output_column="policy.credibility_weighting",
        calculation="df['policy.claim_count'] / (df['policy.claim_count'] + 100)",
        dependencies=["policy.claim_count"],
        description="Policy credibility weighting.",
    ),
    ColumnCalculation(
        name="policy.premium_stat_prem_risk_cost",
        target_table="policy",
        output_column="policy.premium_stat_prem_risk_cost",
        calculation=(
            "df['policy.credibility_weighting'] * df['policy.claims_rated_pred_claim_cost']"
            " + (1 - df['policy.credibility_weighting']) * df['policy.pred_claim_cost']"
        ),
        dependencies=[
            "policy.credibility_weighting",
            "policy.claims_rated_pred_claim_cost",
            "policy.pred_claim_cost",
        ],
        description="Credibility-weighted policy risk cost.",
    ),
    ColumnCalculation(
        name="risk.policy_pred_claim_cost",
        target_table="risk",
        output_column="risk.policy_pred_claim_cost",
        calculation="aggregate(source='policy', source_column='policy.pred_claim_cost', agg_func='sum')",
        dependencies=["policy.pred_claim_cost"],
    ),
    ColumnCalculation(
        name="risk.premium_stat_prem_risk_cost",
        target_table="risk",
        output_column="risk.premium_stat_prem_risk_cost",
        calculation=(
            "aggregate(source='policy', source_column='policy.premium_stat_prem_risk_cost', agg_func='sum')"
            " * df['risk.pred_claim_cost'] / df['risk.policy_pred_claim_cost']"
        ),
        dependencies=["policy.premium_stat_prem_risk_cost", "risk.pred_claim_cost", "risk.policy_pred_claim_cost"],
        description="Policy risk cost allocated to each vehicle.",
    ),
    ColumnCalculation(
        name="risk.premium_stat_prem_administration_expense",
        target_table="risk",
        output_column="risk.premium_stat_prem_administration_expense",
        calculation=f"{expense('expense_administration_fixed')} + {expense('expense_administration_variable')} * {RISK_COST}",
        dependencies=["risk.premium_stat_prem_risk_cost"],
        description="Administration expense component.",
    ),
    ColumnCalculation(
        name="risk.premium_stat_prem_claims_handling_expense",
        target_table="risk",
        output_column="risk.premium_stat_prem_claims_handling_expense",
        calculation=f"{expense('expense_claims_handling_fixed')} + {expense('expense_claims_handling_variable')} * {RISK_COST}",
        dependencies=["risk.premium_stat_prem_risk_cost"],
        description="Claims handling expense component.",
    ),
    ColumnCalculation(
        name="risk.premium_stat_prem_commission_expense",
        target_table="risk",
        output_column="risk.premium_stat_prem_commission_expense",
        calculation=f"{expense('expense_commission_fixed')} + {expense('expense_commission_variable')} * {RISK_COST}",
        dependencies=["risk.premium_stat_prem_risk_cost"],
        description="Commission expense component.",
    ),
    ColumnCalculation(
        name="risk.premium_stat_prem_reinsurance_expense",
        target_table="risk",
        output_column="risk.premium_stat_prem_reinsurance_expense",
        calculation=f"{expense('expense_reinsurance_expense')} * {RISK_COST}",
        dependencies=["risk.premium_stat_prem_risk_cost"],
        description="Reinsurance expense component.",
    ),
    ColumnCalculation(
        name="risk.premium_stat_prem_investment_income",
        target_table="risk",
        output_column="risk.premium_stat_prem_investment_income",
        calculation=f"-{expense('expense_investment_income')} * {RISK_COST}",
        dependencies=["risk.premium_stat_prem_risk_cost"],
        description="Investment income credit.",
    ),
    ColumnCalculation(
        name="risk.premium_stat_prem_target_profit_margin",
        target_table="risk",
        output_column="risk.premium_stat_prem_target_profit_margin",
        calculation=f"{expense('expense_target_profit_margin_percent')} * {RISK_COST}",
        dependencies=["risk.premium_stat_prem_risk_cost"],
        description="Target profit margin component.",
    ),
]

stat_prem_components = [
    "risk.premium_stat_prem_risk_cost",
    "risk.premium_stat_prem_administration_expense",
    "risk.premium_stat_prem_claims_handling_expense",
    "risk.premium_stat_prem_commission_expense",
    "risk.premium_stat_prem_reinsurance_expense",
    "risk.premium_stat_prem_investment_income",
    "risk.premium_stat_prem_target_profit_margin",
]

statistical_premium.append(
    ColumnCalculation(
        name="risk.premium_stat_prem_total",
        target_table="risk",
        output_column="risk.premium_stat_prem_total",
        calculation=" + ".join(f"df['{column}']" for column in stat_prem_components),
        dependencies=stat_prem_components,
        description="Total statistical (technical) premium.",
    )
)

### 5.5 Demand premium

A `RatingTablesLayer` looks up a factor from each of its rating tables and combines them with `formula` (by default, it multiplies them). The key column of each rating table must match a column on the target table. A row with no match takes the table's `default_values` entry.

All factors are 1.0 here, so the demand premium equals the statistical premium. They exist as the levers a pricing team adjusts, or that the price optimisers in `open_rating_engine.optimization` tune.

In [10]:
demand_premium = [
    RatingTablesLayer(
        name="risk.demand_multiplier",
        target_table="risk",
        output_column="risk.demand_multiplier",
        rating_tables={
            "risk_age_band": pd.DataFrame({
                "risk.risk_age_band": ["15-24", "25-34", "35-44", "45-54", "55-64", "65+"],
                "value": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0],
            }),
            "customer_tenure_band": pd.DataFrame({
                "risk.customer_tenure_band": ["0-5", "6-10", "10+"],
                "value": [1.0, 1.0, 1.0],
            }),
        },
        formula="risk_age_band['value'] * customer_tenure_band['value']",
        default_values={"risk_age_band": 1, "customer_tenure_band": 1},
        dependencies=["risk.risk_age_band", "risk.customer_tenure_band"],
        description="Demand multiplier from rating tables.",
    ),
    RatingTablesLayer(
        name="risk.demand_multiplier_individual",
        target_table="risk",
        output_column="risk.demand_multiplier_individual",
        # Placeholder for per-vehicle multipliers: unmatched vehicles take the default of 1.
        rating_tables={
            "risk_number": pd.DataFrame({"risk.risk_number": ["temp"], "value": [1.0]}),
        },
        formula="risk_number['value']",
        default_values={"risk_number": 1},
        description="Individual (per-vehicle) demand multiplier.",
    ),
    ColumnCalculation(
        name="risk.premium_demand_total",
        target_table="risk",
        output_column="risk.premium_demand_total",
        calculation="df['risk.premium_stat_prem_total'] * df['risk.demand_multiplier'] * df['risk.demand_multiplier_individual']",
        dependencies=["risk.premium_stat_prem_total", "risk.demand_multiplier", "risk.demand_multiplier_individual"],
        description="Demand-adjusted premium.",
    ),
]

### 5.6 Cap / cup

Limits the year-on-year premium movement for renewals. The cap (maximum increase) and cup (maximum decrease) come from the catalog by tenure band. New business has no previous premium, so it passes through uncapped.

In [11]:
cap_cup_premium = [
    TableJoin(
        name="policy_assumptions_cap_cup",
        left_table="risk",
        right_table="assumptions_cap_cup",
        right_source="catalog",
        left_on="risk.customer_tenure_band",
        right_on="customer_tenure_band",
        columns=["cap_percentage", "cup_percentage"],
        how="left",
        dependencies=["risk.customer_tenure_band"],
        description="Bring cap/cup percentages onto risk.",
    ),
    ColumnCalculation(
        name="cap_percentage_filled",
        target_table="risk",
        output_column="cap_percentage_filled",
        calculation="df['cap_percentage'].fillna(2)",
        dependencies=["policy_assumptions_cap_cup"],
        description="Fill missing cap percentage.",
    ),
    ColumnCalculation(
        name="cup_percentage_filled",
        target_table="risk",
        output_column="cup_percentage_filled",
        calculation="df['cup_percentage'].fillna(-0.5)",
        dependencies=["policy_assumptions_cap_cup"],
        description="Fill missing cup percentage.",
    ),
    ColumnCalculation(
        name="risk.premium_comm_prem_cap_cup",
        target_table="risk",
        output_column="risk.premium_comm_prem_cap_cup",
        calculation=(
            "np.where("
            "df['risk.previous_gwp'].isna(), "
            "df['risk.premium_demand_total'], "
            "np.maximum("
            "np.minimum(df['risk.premium_demand_total'], df['risk.previous_gwp'] * (1 + df['cap_percentage_filled'])), "
            "df['risk.previous_gwp'] * (1 + df['cup_percentage_filled'])"
            "))"
        ),
        dependencies=["risk.premium_demand_total", "cap_percentage_filled", "cup_percentage_filled"],
        description="Cap/cup the demand premium; new business (null previous_gwp) passes through uncapped.",
    ),
    ColumnCalculation(
        name="risk.premium_final",
        target_table="risk",
        output_column="risk.premium_final",
        calculation="df['risk.premium_comm_prem_cap_cup']",
        dependencies=["risk.premium_comm_prem_cap_cup"],
        description="Final offered premium.",
    ),
]

### 5.7 Final premium

In [12]:
final_premium = [
    ColumnCalculation(
        name="risk.gwp",
        target_table="risk",
        output_column="risk.gwp",
        calculation="df['risk.premium_comm_prem_cap_cup']",
        dependencies=["risk.premium_comm_prem_cap_cup"],
        description="Gross written premium.",
    ),
    ColumnCalculation(
        name="policy.gwp",
        target_table="policy",
        output_column="policy.gwp",
        calculation="aggregate(source='risk', source_column='risk.gwp', agg_func='sum')",
        dependencies=["risk.gwp"],
        description="Gross written premium for the policy.",
    ),
]

### 5.8 Demand model

An H2O GBM predicts the probability that the customer accepts the offered premium. Its features describe the vehicle, its price position against competitors and last year's premium, and one-hot encodings of usage and transaction type. These are built as ordinary layers first, then passed to the model through `column_map` (MOJO column name to target table column).

For a `binomial` model the layer writes the probability of the positive class.

In [13]:
demand_features = [
    ColumnCalculation(
        name="cat_onehot__risk_usage_Private",
        target_table="risk",
        output_column="cat_onehot__risk_usage_Private",
        calculation="(df['risk.vehicle_usage'] == 'Private').astype(int)",
        description="One-hot private vehicle usage.",
    ),
    ColumnCalculation(
        name="cat_onehot__risk_usage_Rideshare",
        target_table="risk",
        output_column="cat_onehot__risk_usage_Rideshare",
        calculation="(df['risk.vehicle_usage'] == 'Rideshare').astype(int)",
        description="One-hot rideshare vehicle usage.",
    ),
    ColumnCalculation(
        name="cat_onehot__transaction_type_RN",
        target_table="risk",
        output_column="cat_onehot__transaction_type_RN",
        calculation="(df['risk.transaction_type'] == 'RN').astype(int)",
        dependencies=["risk.transaction_type"],
        description="One-hot renewal transaction type.",
    ),
    ColumnCalculation(
        name="num__distance_to_competitor_percent",
        target_table="risk",
        output_column="num__distance_to_competitor_percent",
        calculation=(
            "np.where(df['risk.competitor_premiums_average'] != 0, "
            "(df['risk.gwp'] - df['risk.competitor_premiums_average']) / df['risk.competitor_premiums_average'], "
            "np.nan)"
        ),
        dependencies=["risk.gwp"],
        description="Relative distance to competitor premium.",
    ),
    ColumnCalculation(
        name="num__distance_to_competitor_abs",
        target_table="risk",
        output_column="num__distance_to_competitor_abs",
        calculation="np.abs(df['risk.gwp'] - df['risk.competitor_premiums_average'])",
        dependencies=["risk.gwp"],
        description="Absolute distance to competitor premium.",
    ),
    ColumnCalculation(
        name="num__price_change",
        target_table="risk",
        output_column="num__price_change",
        calculation=(
            "np.where(df['risk.previous_gwp'] != 0, "
            "(df['risk.gwp'] - df['risk.previous_gwp']) / df['risk.previous_gwp'], "
            "np.nan)"
        ),
        dependencies=["risk.gwp"],
        description="Relative price change vs previous premium.",
    ),
    ColumnCalculation(
        name="num__gwp",
        target_table="risk",
        output_column="num__gwp",
        calculation="df['risk.gwp']",
        dependencies=["risk.gwp"],
        description="Numeric GWP feature.",
    ),
]

demand_model = demand_features + [
    H2OScoringParameter(
        name="risk.conversion_rate",
        target_table="risk",
        output_column="risk.conversion_rate",
        mojo_artifact_key="motor_demand_mojo",
        genmodel_artifact_key="h2o_gen_model",
        model_type="binomial",
        n_threads=4,
        column_map={
            "num__driver_youngest_age": "risk.driver_youngest_age",
            "num__vehicle_market_value": "risk.vehicle_market_value",
            "num__distance_to_competitor_percent": "num__distance_to_competitor_percent",
            "num__distance_to_competitor_abs": "num__distance_to_competitor_abs",
            "num__price_change": "num__price_change",
            "num__gwp": "num__gwp",
            "cat_onehot__risk_usage_Private": "cat_onehot__risk_usage_Private",
            "cat_onehot__risk_usage_Rideshare": "cat_onehot__risk_usage_Rideshare",
            "cat_onehot__transaction_type_RN": "cat_onehot__transaction_type_RN",
        },
        dependencies=[layer.name for layer in demand_features],
        description="Demand conversion rate scored from an H2O MOJO.",
    ),
]

### 5.9 Reporting

KPIs at the offered premium, and *expected* KPIs weighted by the conversion probability.

In [14]:
reporting = [
    ColumnCalculation(
        name="risk.reporting_previous_gwp",
        target_table="risk",
        output_column="risk.reporting_previous_gwp",
        calculation="df['risk.previous_gwp']",
        description="Previous GWP.",
    ),
    ColumnCalculation(
        name="risk.reporting_offered_gwp",
        target_table="risk",
        output_column="risk.reporting_offered_gwp",
        calculation="df['risk.premium_final']",
        dependencies=["risk.premium_final"],
        description="Offered GWP.",
    ),
    ColumnCalculation(
        name="risk.reporting_offered_technical_premium",
        target_table="risk",
        output_column="risk.reporting_offered_technical_premium",
        calculation="df['risk.premium_stat_prem_total']",
        dependencies=["risk.premium_stat_prem_total"],
        description="Offered technical premium.",
    ),
    ColumnCalculation(
        name="risk.reporting_offered_profit",
        target_table="risk",
        output_column="risk.reporting_offered_profit",
        calculation="df['risk.reporting_offered_gwp'] - df['risk.reporting_offered_technical_premium']",
        dependencies=["risk.reporting_offered_gwp", "risk.reporting_offered_technical_premium"],
        description="Offered profit.",
    ),
    ColumnCalculation(
        name="risk.reporting_offered_loss_ratio",
        target_table="risk",
        output_column="risk.reporting_offered_loss_ratio",
        calculation="df['risk.reporting_offered_technical_premium'] / df['risk.reporting_offered_gwp']",
        dependencies=["risk.reporting_offered_technical_premium", "risk.reporting_offered_gwp"],
        description="Offered loss ratio.",
    ),
    ColumnCalculation(
        name="risk.reporting_offered_profit_margin",
        target_table="risk",
        output_column="risk.reporting_offered_profit_margin",
        calculation="df['risk.reporting_offered_profit'] / df['risk.reporting_offered_gwp']",
        dependencies=["risk.reporting_offered_profit", "risk.reporting_offered_gwp"],
        description="Offered profit margin.",
    ),
    ColumnCalculation(
        name="risk.reporting_offered_premium_change",
        target_table="risk",
        output_column="risk.reporting_offered_premium_change",
        calculation="df['risk.reporting_offered_gwp'] / df['risk.previous_gwp'] - 1",
        dependencies=["risk.reporting_offered_gwp"],
        description="Offered premium change.",
    ),
    ColumnCalculation(
        name="risk.reporting_expected_gwp",
        target_table="risk",
        output_column="risk.reporting_expected_gwp",
        calculation="df['risk.reporting_offered_gwp'] * df['risk.conversion_rate']",
        dependencies=["risk.reporting_offered_gwp", "risk.conversion_rate"],
        description="Expected GWP.",
    ),
    ColumnCalculation(
        name="risk.reporting_expected_technical_premium",
        target_table="risk",
        output_column="risk.reporting_expected_technical_premium",
        calculation="df['risk.reporting_offered_technical_premium'] * df['risk.conversion_rate']",
        dependencies=["risk.reporting_offered_technical_premium", "risk.conversion_rate"],
        description="Expected technical premium.",
    ),
    ColumnCalculation(
        name="risk.reporting_expected_profit",
        target_table="risk",
        output_column="risk.reporting_expected_profit",
        calculation="df['risk.reporting_offered_profit'] * df['risk.conversion_rate']",
        dependencies=["risk.reporting_offered_profit", "risk.conversion_rate"],
        description="Expected profit.",
    ),
    ColumnCalculation(
        name="risk.reporting_expected_profit_margin",
        target_table="risk",
        output_column="risk.reporting_expected_profit_margin",
        calculation="df['risk.reporting_expected_profit'] / df['risk.reporting_expected_gwp']",
        dependencies=["risk.reporting_expected_profit", "risk.reporting_expected_gwp"],
        description="Expected profit margin.",
    ),
]

## 6. Build the pipeline

Register the layers, the catalog and the data structure. The order layers are added in doesn't matter: the pipeline works out the run order from the dependencies, and raises an error if a dependency is missing or circular.

In [15]:
layers = (
    data_prep
    + exposure_rating
    + claims_rating
    + statistical_premium
    + demand_premium
    + cap_cup_premium
    + final_premium
    + demand_model
    + reporting
)

pipeline = PyfuncPricingPipeline(context)

for layer in layers:
    pipeline.add_parameter(layer)

for name, table in catalog.items():
    pipeline.set_catalog(name, table)

pipeline.options["data_structure"] = data_structure

print(f"{len(pipeline.get_layer_names())} layers registered")

57 layers registered


The execution plan groups the layers into **waves**. Every layer in a wave depends only on earlier waves, so the column calculations in a wave run in parallel.

In [16]:
pipeline.print_execution_plan()

Execution Plan

── Wave 1  (parallel — 12 layers) ────────────────────
    1. [COL]  [risk] → risk.transaction_type
         Layer:        risk.transaction_type
         Calculation:  "aggregate(source='policy', source_column='policy.transaction_type', agg_func='first')"
         Dependencies: —
         Description:  Policy transaction type (NB / RN) copied onto each vehicle.
    2. [COL]  [risk] → risk.customer_tenure
         Layer:        risk.customer_tenure
         Calculation:  "aggregate(source='policy', source_column='policy.customer_tenure', agg_func='first')"
         Dependencies: —
         Description:  Customer tenure copied onto each vehicle.
    3. [COL]  [risk] → risk.risk_age_band
         Layer:        risk.risk_age_band
         Calculation:  "pd.cut(df['risk.driver_youngest_age'], bins=list(range(15, 66, 10)) + [np.inf], labels=['15-24', '25-34', '35-44', '45-54', '55-64', '65+'])"
         Dependencies: —
         Description:  Youngest driver age band.
    4. [

`plot_dag()` draws the same graph interactively. It needs the `viz` extra (`plotly`, `networkx`).

In [17]:
pipeline.plot_dag(height=800)

## 7. Score

`load_context` resolves the artifact keys to files, starts the JVM for the H2O layer and loads the MOJO. `predict` then runs every wave and returns the input tables with every computed column added.

In [19]:
pipeline.load_context(context)

result = pipeline.predict(data)

Load Context: Adding Artifact - motor_risk_cost_onnx
Load Context: Adding Artifact - motor_demand_mojo
Load Context: Adding Artifact - h2o_gen_model
Load Context: Starting JVM...
Load Context: {'motor_risk_cost_onnx': 'C:/git/bhavek-jamnadas/open-rating-engine/.examples/models/motor_risk_cost_onnx/pipeline_onnx.onnx', 'motor_demand_mojo': 'C:/git/bhavek-jamnadas/open-rating-engine/.examples/models/motor_demand_h2o/model.zip', 'h2o_gen_model': 'C:/git/bhavek-jamnadas/open-rating-engine/.examples/models/motor_demand_h2o/h2o-genmodel.jar'}, {'environment': 'local', 'java_options': '-Xmx1g'}
Calculating Prices: 100%|██████████| 16/16 [00:06<00:00,  2.47it/s]


### Premium build-up by vehicle

In [20]:
build_up_columns = [
    "risk.policy_number",
    "risk.risk_number",
    "risk.transaction_type",
    "risk.pred_claim_cost",
    *stat_prem_components,
    "risk.premium_stat_prem_total",
    "risk.premium_demand_total",
    "risk.previous_gwp",
    "risk.premium_final",
    "risk.conversion_rate",
]

result["risk"][build_up_columns].set_index("risk.risk_number").T.round(2)

risk.risk_number,cdf6c3a288_NB,0daa59c822_NB,c43da7ccea_NB,dc761835aa_NB,35790b6361_NB,f02b7d5249_NB,56f2a66b49_NB,d1cf4302fc_NB,aae50165e0_NB,5847dd8fb0_NB,...,bdef9b1f98_NB,2053408c04_NB,4641a822bf_NB,8b31927e38_NB,c12b93608c_NB,54ed63b0d0_NB,b58d3a2a8b_NB,de9fce4933_NB,d7ba881d64_NB,736512837f_NB
risk.policy_number,c22591ed41_NB,f44ffa7063_NB,5bcc83eeae_NB,034832413c_NB,d9cd070b29_NB,f1c6554d7e_NB,d302873693_NB,413b20b4d7_NB,cb160a9118_NB,d1287a890e_NB,...,7f85173c6e_NB,4aa7bc3659_NB,193dcc04ab_NB,2323530704_NB,d8af0027e3_NB,27689f51b8_NB,e45b6dd94b_NB,031bd9dd18_NB,25b78dcae6_NB,1eaab3de07_NB
risk.transaction_type,NB,NB,NB,NB,NB,NB,NB,NB,NB,NB,...,NB,NB,NB,NB,NB,NB,NB,NB,NB,NB
risk.pred_claim_cost,394.385406,1557.219116,952.85675,1880.842773,2409.525146,478.412384,2175.90918,797.551025,1607.27771,1964.705444,...,479.46405,377.503754,769.866516,563.717896,548.16626,494.194611,701.444031,631.74469,482.1828,454.844788
risk.premium_stat_prem_risk_cost,407.77724,1522.487258,934.396794,1831.772349,2346.434569,481.029052,2150.579468,772.91221,1544.285167,1969.570488,...,489.953932,379.495024,878.412729,551.820405,564.093579,508.135487,727.24007,637.696146,472.42793,451.640114
risk.premium_stat_prem_administration_expense,83.155545,105.449745,93.687936,111.635447,121.928691,84.620581,118.011589,90.458244,105.885703,114.39141,...,84.799079,82.5899,92.568255,86.036408,86.281872,85.16271,89.544801,87.753923,84.448559,84.032802
risk.premium_stat_prem_claims_handling_expense,20.388862,76.124363,46.71984,91.588617,117.321728,24.051453,107.528973,38.645611,77.214258,98.478524,...,24.497697,18.974751,43.920636,27.59102,28.204679,25.406774,36.362003,31.884807,23.621397,22.582006
risk.premium_stat_prem_commission_expense,20.388862,76.124363,46.71984,91.588617,117.321728,24.051453,107.528973,38.645611,77.214258,98.478524,...,24.497697,18.974751,43.920636,27.59102,28.204679,25.406774,36.362003,31.884807,23.621397,22.582006
risk.premium_stat_prem_reinsurance_expense,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
risk.premium_stat_prem_investment_income,-20.388862,-76.124363,-46.71984,-91.588617,-117.321728,-24.051453,-107.528973,-38.645611,-77.214258,-98.478524,...,-24.497697,-18.974751,-43.920636,-27.59102,-28.204679,-25.406774,-36.362003,-31.884807,-23.621397,-22.582006
risk.premium_stat_prem_target_profit_margin,40.777724,152.248726,93.439679,183.177235,234.643457,48.102905,215.057947,77.291221,154.428517,196.957049,...,48.995393,37.949502,87.841273,55.18204,56.409358,50.813549,72.724007,63.769615,47.242793,45.164011


### Policy summary

The renewal's offered premium is capped relative to last year's, while new business takes the uncapped demand premium. The credibility weight shows how much of each policy's risk cost comes from its own claims history.

In [21]:
result["policy"][
    [
        "policy.policy_number",
        "policy.transaction_type",
        "policy.vehicle_count",
        "policy.claim_count",
        "policy.credibility_weighting",
        "policy.pred_claim_cost",
        "policy.claims_rated_pred_claim_cost",
        "policy.premium_stat_prem_risk_cost",
        "policy.previous_gwp",
        "policy.gwp",
    ]
].set_index("policy.policy_number").T.round(3)

policy.policy_number,000622c34d_NB,000b476cb9_NB,001cc64177_NB,00216c3400_NB,002ae828a3_NB,00324a9cbf_NB,0045423929_NB,0049a3eff2_NB,00715d48e0_NB,0087da1007_NB,...,ff3725681d_NB,ff40457ae0_NB,ff4690f946_NB,ff4cffa0bd_NB,ff6c3cbca8_NB,ff7da2ad20_NB,ff844bc9a3_NB,ffad349f06_NB,ffb0d2d93f_NB,ffbfb10e14_NB
policy.transaction_type,NB,NB,NB,NB,NB,NB,NB,NB,NB,NB,...,NB,NB,NB,NB,NB,NB,NB,NB,NB,NB
policy.vehicle_count,9,11,9,9,8,7,8,5,5,5,...,9,15,8,13,10,4,11,6,4,8
policy.claim_count,11,5,5,6,5,2,6,5,1,7,...,4,9,7,6,9,2,7,4,5,8
policy.credibility_weighting,0.099099,0.047619,0.047619,0.056604,0.047619,0.019608,0.056604,0.047619,0.009901,0.065421,...,0.038462,0.082569,0.065421,0.056604,0.082569,0.019608,0.065421,0.038462,0.047619,0.074074
policy.pred_claim_cost,10046.543945,9958.873047,7801.119141,6444.946289,5464.466309,5617.440918,7960.857422,3283.666748,4754.320312,4036.340576,...,11237.09375,14501.082031,7299.357422,13914.893555,11266.407227,2284.999512,9168.212891,3987.905762,2186.928467,7617.189453
policy.claims_rated_pred_claim_cost,12370.708243,5022.719942,6650.359058,4942.800824,3863.47664,2506.812797,5649.013074,4257.042893,683.814089,5218.130233,...,4941.429687,10476.658038,12720.824782,5466.510028,8869.146175,1433.767406,5813.985963,2839.904422,4899.192315,6128.198788
policy.premium_stat_prem_risk_cost,10276.866533,9723.818137,7746.321041,6359.919187,5388.228705,5556.44821,7829.998308,3330.017993,4714.018271,4113.653918,...,10994.952825,14168.790142,7654.032857,13436.683166,11068.468241,2268.308686,8948.777484,3943.751864,2316.083888,7506.893848
policy.previous_gwp,6918.158414,3521.83,1433.731635,5054.22,1168.87,2988.07,4250.52,3721.322193,2575.912927,4157.975916,...,6359.354441,11061.078829,6428.11,8547.248971,7237.528082,675.79,6497.629384,3285.37,1569.29,3731.69
policy.gwp,13068.821445,12323.010774,10121.682486,8447.256082,7028.884958,7089.475295,10234.176845,4859.394687,6001.82177,5211.747143,...,12558.629079,18100.530022,10267.286771,17127.756862,14213.811265,3083.260398,11289.519501,5449.349833,3029.334054,9360.13921


### Reporting KPIs

In [22]:
reporting_columns = ["risk.risk_number"] + [layer.output_column for layer in reporting]

result["risk"][reporting_columns].set_index("risk.risk_number").round(3)

,risk.reporting_previous_gwp,risk.reporting_offered_gwp,risk.reporting_offered_technical_premium,risk.reporting_offered_profit,risk.reporting_offered_loss_ratio,risk.reporting_offered_profit_margin,risk.reporting_offered_premium_change,risk.reporting_expected_gwp,risk.reporting_expected_technical_premium,risk.reporting_expected_profit,risk.reporting_expected_profit_margin
risk.risk_number,,,,,,,,,,,
cdf6c3a288_NB,NaN,552.099,552.099,0.000,1.000,0.000,NaN,444.290,444.290,0.000,0.000
0daa59c822_NB,NaN,1856.310,1856.310,0.000,1.000,0.000,NaN,235.935,235.935,0.000,0.000
c43da7ccea_NB,NaN,1168.244,1168.244,0.000,1.000,0.000,NaN,222.001,222.001,0.000,0.000
dc761835aa_NB,NaN,2218.174,2218.174,0.000,1.000,0.000,NaN,42.587,42.587,0.000,0.000
35790b6361_NB,NaN,2820.328,2820.328,0.000,1.000,0.000,NaN,1048.019,1048.019,0.000,0.000
...,...,...,...,...,...,...,...,...,...,...,...
54ed63b0d0_NB,522.697,669.519,669.519,0.000,1.000,0.000,0.281,214.584,214.584,0.000,0.000
b58d3a2a8b_NB,NaN,925.871,925.871,0.000,1.000,0.000,NaN,149.579,149.579,0.000,0.000
de9fce4933_NB,NaN,821.104,821.104,0.000,1.000,0.000,NaN,152.260,152.260,0.000,0.000
